In [14]:
import sys
print(sys.executable)
print(sys.version)
print(sys.path)


c:\Program Files\Python313\python.exe
3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]
['c:\\Program Files\\Python313\\python313.zip', 'c:\\Program Files\\Python313\\DLLs', 'c:\\Program Files\\Python313\\Lib', 'c:\\Program Files\\Python313', '', 'C:\\Users\\DRVACAE\\AppData\\Roaming\\Python\\Python313\\site-packages', 'C:\\Users\\DRVACAE\\AppData\\Roaming\\Python\\Python313\\site-packages\\win32', 'C:\\Users\\DRVACAE\\AppData\\Roaming\\Python\\Python313\\site-packages\\win32\\lib', 'C:\\Users\\DRVACAE\\AppData\\Roaming\\Python\\Python313\\site-packages\\pythonwin', 'c:\\Program Files\\Python313\\Lib\\site-packages']


In [8]:
import numpy as np
import pdfplumber
import pandas as pd
import re

In [21]:
def cargar_pdf(ruta_pdf: str):
    f = open(ruta_pdf, "rb")
    pdf = pdfplumber.open(f)
    return pdf, f

def extraer_texto_paginas(pdf) -> str:
    texto = ""
    for p in pdf.pages:
        contenido = p.extract_text()
        if contenido:
            texto += contenido + "\n"
    return texto

def limpiar_lineas(texto: str) -> list:
    lineas = [line.strip() for line in texto.split("\n") if line.strip()]
    return lineas

def extraer_transacciones(lineas: list) -> pd.DataFrame:
    patron = re.compile(
        r"(\d{2}/\d{2})\s+(\d+)\s+(.*?)\s+[SQC]\s+(\d+,\d+)"
    )

    registros = []
    for linea in lineas:
        match = patron.search(linea)
        if match:
            fecha, vale, establecimiento, valor = match.groups()
            registros.append({
                "fecha": fecha,
                "vale": vale,
                "establecimiento": establecimiento,
                "valor": float(valor.replace(",", "."))  # convertir a float
            })

    return pd.DataFrame(registros)

def cargar_estado_cuenta(ruta_pdf: str) -> pd.DataFrame:
    pdf, f = cargar_pdf(ruta_pdf)
    texto = extraer_texto_paginas(pdf)
    lineas = limpiar_lineas(texto)
    df = extraer_transacciones(lineas)
    pdf.close()
    f.close()
    return df

if __name__ == "__main__":
    ruta = "d:/proyectos_py/presupuesto/data/IDVIXXXXXXXX6511.pdf"
    df = cargar_estado_cuenta(ruta)
    print(df)



    fecha     vale     establecimiento  valor
0   05/06  3699772          UBER RIDES   4.65
1   06/06  3699773          UBER RIDES   3.27
2   11/06  4568401          UBER RIDES   3.20
3   15/06  5081154          UBER RIDES   3.15
4   08/06  4014174        PAF EL CISNE  40.58
5   27/05  2719116         MEET2GO PGX  55.00
6   29/05  2912146     CRUNCHYROLL.COM   3.99
7   02/06  3479551             SHOP IN   6.89
8   02/06  3479552             SHOP IN  20.29
9   04/06  3591601     Zapping Ecuador  11.90
10  16/06  5123625  Spotify P439474561   3.49


In [ ]:
df.to_excel("estado_cuenta.xlsx", index=False)